In [6]:
from google.adk.agents.llm_agent import Agent
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
# Cell 3
import google.auth
from google.adk.telemetry import google_cloud
from google.adk.telemetry.setup import maybe_set_otel_providers

# Resolve credentials and project ID explicitly
credentials, project_id = google.auth.default()
print(f"Project ID: {project_id}")

# Build OTel resource with the project ID so Cloud Trace knows where to export
resource = google_cloud.get_gcp_resource(project_id=project_id)

# Get GCP exporters configuration
hooks = google_cloud.get_gcp_exporters(
    enable_cloud_tracing=True,
    google_auth=(credentials, project_id),
)

# Initialize and set global OTel providers
maybe_set_otel_providers(otel_hooks_to_setup=[hooks], otel_resource=resource)

print("✅ Cloud Trace initialized")

Project ID: dev-mq-tech-transfer
✅ Cloud Trace initialized


In [8]:
import asyncio
from google.adk.runners import InMemoryRunner
from google.adk.runners import Runner

from google.genai.types import Content, Part
import asyncio
import os

from google.genai import types
from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts.in_memory_artifact_service import InMemoryArtifactService # Optional
from google.adk.planners import BasePlanner, BuiltInPlanner, PlanReActPlanner
from google.adk.models import LlmRequest

from google.genai.types import ThinkingConfig
from google.genai.types import GenerateContentConfig


APP_NAME = "city_time_app"
USER_ID = "prakash.kc"

In [9]:
# from google.adk.agents import Agent

# root_agent = Agent(
#     name="capital_agent",
#     model="gemini-2.5-flash",
#     description="Answers geography questions.",
#     instruction="Answer the user's question directly and concisely.",
# )

# print("✅ Agent created")

# Mock tool implementation
def get_current_time(city: str) -> dict:
    """Returns the current time in a specified city."""
    return {"status": "success", "city": city, "time": "10:30 AM"}

root_agent = Agent(
    model=os.environ.get("MODEL_NAME", "gemini-2.5-flash"),
    name='root_agent',
    description="Tells the current time in a specified city.",
    instruction="You are a helpful assistant that tells the current time in cities. Use the 'get_current_time' tool for this purpose.",
    tools=[get_current_time],
)

In [10]:
# Cell 5
import asyncio
from google.adk.runners import InMemoryRunner
from google.genai.types import Content, Part

async def ask(question: str):
    runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)

    session = await runner.session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
    )

    user_message = Content(parts=[Part(text=question)])

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=user_message,
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print("Agent:", part.text)



In [11]:
await ask("What is the time in London?")

c:\Users\L137860\OneDrive - Eli Lilly and Company\Playground\gcp_trace_exp\gcp_trace_test\.venv\Lib\site-packages\google\adk\tools\function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


Agent: The current time in London is 10:30 AM.
